# 점자블록 영역 학습 (무료 코랩)

1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. `braille_seg.zip`(또는 압축을 푼 `braille_seg` 폴더)을 내 드라이브 맨 위(`MyDrive/`)에 올려 두기 (AI Hub 데이터라 공유 금지)
3. 위에서부터 차례로 실행. 끊기면 다시 위에서부터 실행하면 마지막 저장 지점부터 이어서 학습해요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

In [ ]:
!pip -q install ultralytics==8.4.174
# 드라이브에 zip으로 올렸든 폴더째 올렸든, 학습은 빠른 로컬 디스크(/content)에서 해요
!rm -rf /content/braille_seg
!if [ -f /content/drive/MyDrive/braille_seg.zip ]; then unzip -qo /content/drive/MyDrive/braille_seg.zip -d /content/; else cp -r /content/drive/MyDrive/braille_seg /content/; fi
!for d in images/train images/val labels/train labels/val; do echo $d $(ls /content/braille_seg/$d | wc -l); done  # 3137 / 838 / 3137 / 838

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/gilnun_braille'  # 결과는 드라이브에 저장돼서 세션이 끊겨도 남아요
NAME = 'yolo11n_braille'
last = Path(PROJECT) / NAME / 'weights' / 'last.pt'

if last.exists():
    YOLO(str(last)).train(resume=True)
else:
    YOLO('yolo11n-seg.pt').train(
        data='/content/braille_seg/data.yaml', imgsz=640, epochs=100, batch=16,
        patience=20, project=PROJECT, name=NAME, exist_ok=True,
        fliplr=0.5, degrees=5,  # 휴대폰을 살짝 기울여 들고 걷는 경우
    )

In [ ]:
m = YOLO(f'{PROJECT}/{NAME}/weights/best.pt')
r = m.val(data='/content/braille_seg/data.yaml', imgsz=640)
print('mask mAP50:', r.seg.map50, ' mask mAP50-95:', r.seg.map)

학습이 끝나면 드라이브의 `gilnun_braille/yolo11n_braille/weights/best.pt`를 Mac으로 내려받아 주세요. Core ML 변환은 Mac에서 해요.